In [ ]:
# Trendyol Datathon — EDA

Bu notebook şu başlıklara bakar:

1. Veri kalitesi  
2. Arama terimleri  
3. Ürün kataloğu  
4. Pozitif çiftlerin yapısı  
5. Train–test farkı  
6. Negatif üretim kontrolü  

**Çalıştırma:** `notebooks/` klasöründen açın. İlk hücrede `USE_MINI = True` yaparsanız mini pratik verisi kullanılır.

In [ ]:
from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- AYARLAR ---
USE_MINI = False          # True → examples/mini_pratik
SAMPLE_SUBMISSION = 200_000  # submission çok büyük; hız için örnek (None = tamamı)
RANDOM_STATE = 42

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "examples/mini_pratik" if USE_MINI else ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"

print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)

In [ ]:
# --- VERİ YÜKLEME ---
items = pd.read_csv(DATA_DIR / "items.csv", dtype=str)
terms = pd.read_csv(DATA_DIR / "terms.csv", dtype=str)
training = pd.read_csv(DATA_DIR / "training_pairs.csv", dtype=str)
submission = pd.read_csv(DATA_DIR / "submission_pairs.csv", dtype=str)

if SAMPLE_SUBMISSION and len(submission) > SAMPLE_SUBMISSION:
    submission = submission.sample(n=SAMPLE_SUBMISSION, random_state=RANDOM_STATE)

# Negatif üretim çıktısı (varsa)
neg_path = ARTIFACTS_DIR / "train_with_negatives.csv"
if USE_MINI:
    neg_path = DATA_DIR / "train_with_negatives.csv"

train_neg = pd.read_csv(neg_path, dtype=str) if neg_path.exists() else None
audit = pd.read_csv(ARTIFACTS_DIR / "negative_audit_sample.csv", dtype=str) if (ARTIFACTS_DIR / "negative_audit_sample.csv").exists() else None

print(f"items        : {len(items):,}")
print(f"terms        : {len(terms):,}")
print(f"training     : {len(training):,}")
print(f"submission   : {len(submission):,}" + (f" (sample)" if SAMPLE_SUBMISSION else ""))
print(f"train_neg    : {len(train_neg):,}" if train_neg is not None else "train_neg    : yok")
print(f"audit sample : {len(audit):,}" if audit is not None else "audit sample : yok")

## 1. Veri kalitesi

In [ ]:
def missing_report(df, name):
  miss = df.isna().copy()
  for col in df.columns:
    s = df[col].fillna("").astype(str).str.strip()
    miss[col] = miss[col] | s.eq("") | s.str.lower().isin(["nan", "none", "null"])
  out = miss.sum().sort_values(ascending=False)
  out = out[out > 0]
  print(f"\n=== {name}: eksik/boş ===")
  if out.empty:
    print("Eksik yok")
  else:
    print((out / len(df) * 100).round(2).astype(str) + "%  (" + out.astype(str) + " adet)")

for df, name in [(items, "items"), (terms, "terms"), (training, "training"), (submission, "submission")]:
  missing_report(df, name)

# Tekrarlı kayıtlar
print("\n=== Tekrarlı kayıtlar ===")
print(f"items item_id duplicate     : {items['item_id'].duplicated().sum():,}")
print(f"terms term_id duplicate     : {terms['term_id'].duplicated().sum():,}")
print(f"training (term,item) dup    : {training.duplicated(['term_id','item_id']).sum():,}")
print(f"submission (term,item) dup: {submission.duplicated(['term_id','item_id']).sum():,}")
print(f"training id duplicate       : {training['id'].duplicated().sum():,}")

# Geçersiz ID
valid_terms = set(terms["term_id"])
valid_items = set(items["item_id"])

bad_train_term = ~training["term_id"].isin(valid_terms)
bad_train_item = ~training["item_id"].isin(valid_items)
bad_sub_term = ~submission["term_id"].isin(valid_terms)
bad_sub_item = ~submission["item_id"].isin(valid_items)

print("\n=== Geçersiz term_id / item_id ===")
print(f"training geçersiz term_id : {bad_train_term.sum():,}")
print(f"training geçersiz item_id : {bad_train_item.sum():,}")
print(f"submission geçersiz term_id: {bad_sub_term.sum():,}")
print(f"submission geçersiz item_id: {bad_sub_item.sum():,}")

if "label" in training.columns:
  print("\n=== training label dağılımı ===")
  print(training["label"].value_counts(dropna=False))

In [ ]:
# Boş veya aşırı uzun metinler
def text_length_stats(series, name):
  lengths = series.fillna("").astype(str).str.len()
  empty = (lengths == 0).sum()
  print(f"\n{name}")
  print(f"  boş              : {empty:,} ({empty/len(series)*100:.2f}%)")
  print(f"  uzunluk median   : {lengths.median():.0f}")
  print(f"  uzunluk p95      : {lengths.quantile(0.95):.0f}")
  print(f"  uzunluk max      : {lengths.max():.0f}")
  print(f"  >200 karakter    : {(lengths > 200).sum():,}")
  print(f"  >500 karakter    : {(lengths > 500).sum():,}")
  return lengths

q_len = text_length_stats(terms["query"], "terms.query")
t_len = text_length_stats(items["title"], "items.title")
a_len = text_length_stats(items["attributes"], "items.attributes")

# Uzun metin örnekleri
print("\nEn uzun 5 sorgu:")
display(terms.loc[q_len.nlargest(5).index, ["term_id", "query"]])

print("\nEn uzun 5 başlık:")
display(items.loc[t_len.nlargest(5).index, ["item_id", "title"]])

## 2. Arama terimleri

In [ ]:
# En sık sorgular (birebir aynı metin)
print("=== En sık 20 sorgu (query metni) ===")
print(terms["query"].value_counts().head(20))

# Sorgu uzunlukları (kelime sayısı)
terms_eda = terms.copy()
terms_eda["query_len_char"] = terms_eda["query"].fillna("").str.len()
terms_eda["query_len_word"] = terms_eda["query"].fillna("").str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
terms_eda["query_len_word"].hist(bins=30, ax=axes[0], edgecolor="white")
axes[0].set_title("Sorgu kelime sayısı")
axes[0].set_xlabel("kelime")

terms_eda["query_len_char"].hist(bins=30, ax=axes[1], edgecolor="white")
axes[1].set_title("Sorgu karakter sayısı")
axes[1].set_xlabel("karakter")
plt.tight_layout()
plt.show()

print(terms_eda[["query_len_word", "query_len_char"]].describe().round(1))

In [ ]:
# Türkçe karaktersiz yazım / ASCII fold farkı
TR_CHARS = set("çğıöşüÇĞİÖŞÜ")
ASCII_MAP = str.maketrans({"ç":"c","ğ":"g","ı":"i","ö":"o","ş":"s","ü":"u","Ç":"c","Ğ":"g","İ":"i","Ö":"o","Ş":"s","Ü":"u"})

def has_turkish_char(text):
  return any(ch in TR_CHARS for ch in str(text))

def ascii_fold(text):
  return str(text).lower().translate(ASCII_MAP)

terms_eda["has_tr_char"] = terms_eda["query"].map(has_turkish_char)
terms_eda["ascii_fold"] = terms_eda["query"].map(ascii_fold)
terms_eda["fold_changed"] = terms_eda["query"].str.lower() != terms_eda["ascii_fold"]

# fold sonrası çakışan farklı sorgular (yazım varyantı olabilir)
fold_groups = terms_eda.groupby("ascii_fold")["query"].nunique().sort_values(ascending=False)
multi_variant = fold_groups[fold_groups > 1]

print(f"Türkçe karakter içeren sorgu : {terms_eda['has_tr_char'].sum():,} ({terms_eda['has_tr_char'].mean()*100:.1f}%)")
print(f"ASCII fold ile değişen sorgu : {terms_eda['fold_changed'].sum():,}")
print(f"Birden fazla yazım varyantı  : {len(multi_variant):,} fold grubu")

print("\nÖrnek varyant grupları:")
for fold, _ in multi_variant.head(8).items():
  variants = terms_eda.loc[terms_eda["ascii_fold"] == fold, "query"].unique()[:5]
  print(f"  [{fold}] → {list(variants)}")

In [ ]:
# Sorgu içeriği: marka, renk, cinsiyet, yaş, sayı
COLORS = {"siyah","beyaz","kırmızı","kirmizi","mavi","lacivert","bej","krem","haki","bordo","gri","yeşil","yesil","pembe","mor","turuncu","kahverengi","ekru"}
GENDERS = {"kadın","kadin","bayan","kız","kiz","erkek","unisex"}
AGES = {"bebek","yenidoğan","yenidogan","çocuk","cocuk","yetişkin","yetiskin"}
GENERIC = {"elbise","çanta","canta","bot","ayakkabı","ayakkabi","gömlek","gomlek","mont","telefon","kulaklık","kulaklik","pantolon","kazak","tişört","tisort"}

def token_set(text):
  return set(re.findall(r"\w+", str(text).lower()))

def flag_queries(frame):
  out = frame.copy()
  out["tokens"] = out["query"].map(token_set)
  out["has_color"] = out["tokens"].map(lambda t: bool(t & COLORS))
  out["has_gender"] = out["tokens"].map(lambda t: bool(t & GENDERS))
  out["has_age"] = out["tokens"].map(lambda t: bool(t & AGES))
  out["has_number"] = out["query"].str.contains(r"\d", regex=True, na=False)
  out["is_generic"] = out["tokens"].map(lambda t: len(t) <= 2 and bool(t) and t.issubset(GENERIC))
  out["token_count"] = out["tokens"].map(len)
  return out

terms_flags = flag_queries(terms_eda)

flags = ["has_color", "has_gender", "has_age", "has_number", "is_generic"]
print("=== Sorgu içerik bayrakları ===")
for f in flags:
  print(f"{f:15s}: {terms_flags[f].sum():,} ({terms_flags[f].mean()*100:.1f}%)")

print("\nÖrnek marka içeren sorgular (2+ kelime, generic değil):")
brandish = terms_flags[(terms_flags["token_count"] >= 2) & (~terms_flags["is_generic"])].head(10)
display(brandish[["term_id", "query"]])

## 3. Ürün kataloğu

In [ ]:
items_eda = items.copy()
items_eda["top_category"] = items_eda["category"].fillna("").str.split("/").str[0]
items_eda["leaf_category"] = items_eda["category"].fillna("").str.split("/").str[-1]
items_eda["category_depth"] = items_eda["category"].fillna("").str.count("/") + 1

print("=== Kategori hiyerarşisi ===")
print(f"Üst kategori sayısı : {items_eda['top_category'].nunique():,}")
print(f"Yaprak kategori     : {items_eda['leaf_category'].nunique():,}")
print(f"Tam kategori yolu   : {items_eda['category'].nunique():,}")
print(f"Derinlik median     : {items_eda['category_depth'].median():.0f}")

print("\nTop 15 üst kategori:")
print(items_eda["top_category"].value_counts().head(15))

fig, ax = plt.subplots(figsize=(10, 4))
items_eda["category_depth"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Kategori derinliği dağılımı")
ax.set_xlabel("seviye sayısı")
plt.tight_layout()
plt.show()

In [ ]:
UNKNOWN = {"", "unknown", "bilinmiyor", "none", "nan", "null", "-"}

def unknown_rate(series):
  s = series.fillna("").astype(str).str.lower().str.strip()
  return s.isin(UNKNOWN).mean()

print("=== Marka / gender / age_group ===")
print("Marka sayısı:", items_eda["brand"].nunique())
print(items_eda["brand"].value_counts().head(10))

for col in ["brand", "gender", "age_group"]:
  print(f"\n{col} unknown oranı: {unknown_rate(items_eda[col])*100:.2f}%")
  print(items_eda[col].fillna("").replace("", "unknown").value_counts().head(8))

# Başlık ve attributes doluluk
for col in ["title", "attributes", "category", "brand"]:
  filled = ~items_eda[col].fillna("").astype(str).str.strip().isin(UNKNOWN)
  print(f"{col:12s} dolu: {filled.mean()*100:.2f}%")

## 4. Pozitif çiftlerin yapısı

In [ ]:
pos = training.merge(terms, on="term_id", how="left").merge(items_eda, on="item_id", how="left", suffixes=("_term", "_item"))

# Bir sorgunun kaç pozitif ürünü var?
pos_per_term = pos.groupby("term_id").size()
print("=== Pozitif / sorgu ===")
print(pos_per_term.describe().round(2))
print(f"Sadece 1 pozitifi olan sorgu: {(pos_per_term == 1).sum():,} ({(pos_per_term == 1).mean()*100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
pos_per_term.clip(upper=pos_per_term.quantile(0.99)).hist(bins=40, ax=ax, edgecolor="white")
ax.set_title("Sorgu başına pozitif sayısı (p99 kırpılmış)")
plt.tight_layout()
plt.show()

# Bir ürün kaç sorguyla eşleşiyor?
pos_per_item = pos.groupby("item_id").size()
print("\n=== Pozitif / ürün ===")
print(pos_per_item.describe().round(2))
print(f"Sadece 1 sorguda geçen ürün: {(pos_per_item == 1).sum():,} ({(pos_per_item == 1).mean()*100:.1f}%)")

In [ ]:
# Aynı sorgudaki pozitifler hangi kategorilerde?
term_cat_diversity = pos.groupby("term_id")["top_category"].nunique()
print("=== Sorgu başına kaç farklı üst kategori? ===")
print(term_cat_diversity.value_counts().sort_index().head(10))

multi_cat_terms = term_cat_diversity[term_cat_diversity > 1].index[:5]
print("\nÖrnek: birden fazla üst kategorili sorgular")
for tid in multi_cat_terms:
  q = pos.loc[pos["term_id"] == tid, "query"].iloc[0]
  cats = pos.loc[pos["term_id"] == tid, "top_category"].value_counts().head(5)
  print(f"\n  {q}")
  print(cats.to_string())

# Generic vs spesifik sorgular
term_level = pos[["term_id", "query"]].drop_duplicates()
term_flags = flag_queries(term_level)
generic_terms = set(term_flags.loc[term_flags["is_generic"], "term_id"])

print("\n=== Generic vs spesifik ===")
print(f"Generic sorgu sayısı  : {len(generic_terms):,}")
print(f"Spesifik sorgu sayısı : {term_level['term_id'].nunique() - len(generic_terms):,}")
print(f"Generic pozitif çift  : {pos['term_id'].isin(generic_terms).sum():,}")
print(f"Spesifik pozitif çift : {(~pos['term_id'].isin(generic_terms)).sum():,}")

if generic_terms:
  print("\nGeneric sorgu örnekleri:")
  display(term_flags[term_flags["is_generic"]][["term_id", "query"]].head(10))

## 5. Train–test farkı

In [ ]:
train_terms = set(training["term_id"])
train_items = set(training["item_id"])

sub_terms = set(submission["term_id"])
sub_items = set(submission["item_id"])

new_terms = sub_terms - train_terms
new_items = sub_items - train_items
seen_term_pairs = submission["term_id"].isin(train_terms)
seen_item_pairs = submission["item_id"].isin(train_items)
both_seen = seen_term_pairs & seen_item_pairs

print("=== Train–test örtüşme ===")
print(f"Submission benzersiz term : {len(sub_terms):,}")
print(f"Submission benzersiz item : {len(sub_items):,}")
print(f"Train'de görülen term oranı (çift bazında): {seen_term_pairs.mean()*100:.2f}%")
print(f"Train'de görülen item oranı (çift bazında): {seen_item_pairs.mean()*100:.2f}%")
print(f"Her ikisi de görülmüş çift oranı      : {both_seen.mean()*100:.2f}%")
print(f"Yeni term (hiç train'de yok)          : {len(new_terms):,} ({len(new_terms)/len(sub_terms)*100:.2f}%)")
print(f"Yeni item (hiç train'de yok)          : {len(new_items):,} ({len(new_items)/len(sub_items)*100:.2f}%)")

# Sorgu metni bazında (aynı query farklı term_id olabilir)
train_queries = set(terms.loc[terms["term_id"].isin(train_terms), "query"].str.lower())
sub_query_map = submission.merge(terms, on="term_id", how="left")
sub_query_seen = sub_query_map["query"].str.lower().isin(train_queries)
print(f"\nSubmission çiftlerinde query metni train'de var: {sub_query_seen.mean()*100:.2f}%")

In [ ]:
# Kategori dağılım kayması (train pozitif vs submission)
sub_enriched = submission.merge(terms, on="term_id").merge(items_eda, on="item_id", how="left")

train_cat = pos["top_category"].value_counts(normalize=True).head(15)
sub_cat = sub_enriched["top_category"].value_counts(normalize=True).head(15)

compare = pd.DataFrame({
  "train_pos_share": train_cat,
  "submission_share": sub_cat,
}).fillna(0)
compare["diff"] = compare["submission_share"] - compare["train_pos_share"]

print("=== Top kategori oran farkı (submission - train) ===")
display(compare.sort_values("diff", key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
compare[["train_pos_share", "submission_share"]].head(10).plot(kind="bar", ax=ax)
ax.set_title("Top 10 üst kategori: train pozitif vs submission")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Negatif üretim kontrolü

`artifacts/train_with_negatives.csv` veya mini pratik çıktısı yüklü olmalı.

In [ ]:
if train_neg is None:
  print("train_with_negatives.csv bulunamadı. Önce negatif script'ini çalıştırın.")
else:
  neg = train_neg[train_neg["label"].astype(str) == "0"].copy()
  print(f"Toplam satır: {len(train_neg):,} | negatif: {len(neg):,}")
  print("\n=== Strateji dağılımı ===")
  print(train_neg["negative_type"].value_counts())
  if "negative_confidence" in train_neg.columns:
    print("\n=== Güven dağılımı ===")
    print(neg.groupby(["negative_type", "negative_confidence"]).size())

  print("\n=== Her stratejiden 2 örnek ===")
  for ntype in neg["negative_type"].dropna().unique():
    sample = neg[neg["negative_type"] == ntype].sample(min(2, (neg["negative_type"] == ntype).sum()), random_state=RANDOM_STATE)
    print(f"\n--- {ntype} ---")
    display(sample[["query", "title", "category", "negative_confidence"]])

In [ ]:
# Şüpheli false-negative: sorgu kelimelerinin çoğu ürün metninde geçiyor
STRONG_MATCH_RATIO = 0.75
GENERIC_PRODUCT_QUERIES = GENERIC  # yukarıdaki hücreden

def query_match_ratio(query, item_text):
  qt = token_set(query)
  if not qt:
    return 0.0
  it = token_set(item_text)
  return len(qt & it) / len(qt)

def is_generic_query(query):
  t = token_set(query)
  return len(t) <= 2 and bool(t) and t.issubset(GENERIC_PRODUCT_QUERIES)

def has_strong_match(query, title, category="", attributes=""):
  item_text = " ".join([str(title), str(category), str(attributes)])
  ratio = query_match_ratio(query, item_text)
  if is_generic_query(query):
    return ratio >= 0.60
  return len(token_set(query)) >= 2 and ratio >= STRONG_MATCH_RATIO

if train_neg is not None:
  neg = train_neg[train_neg["label"].astype(str) == "0"].copy()
  neg["match_ratio"] = neg.apply(
    lambda r: query_match_ratio(r["query"], " ".join([r["title"], r["category"], r["attributes"]])),
    axis=1,
  )
  neg["strong_match"] = neg.apply(
    lambda r: has_strong_match(r["query"], r["title"], r["category"], r["attributes"]),
    axis=1,
  )

  print("=== Şüpheli false-negative oranı ===")
  overall = neg["strong_match"].mean()
  print(f"Tüm negatiflerde strong_match: {overall*100:.2f}%")

  by_type = neg.groupby("negative_type")["strong_match"].mean().sort_values(ascending=False)
  print("\nStrateji bazında:")
  print((by_type * 100).round(2).astype(str) + "%")

  suspicious = neg[neg["strong_match"]].sort_values("match_ratio", ascending=False)
  print(f"\nŞüpheli örnek sayısı: {len(suspicious):,}")
  display(suspicious[["negative_type", "negative_confidence", "match_ratio", "query", "title"]].head(15))

In [ ]:
# Hangi sorgu türünde hangi strateji daha güvenli?
if train_neg is not None:
  neg = train_neg[train_neg["label"].astype(str) == "0"].copy()

  if "strong_match" not in neg.columns:
    neg["strong_match"] = neg.apply(
      lambda r: has_strong_match(r["query"], r["title"], r["category"], r["attributes"]),
      axis=1,
    )

  neg = neg.merge(
    terms_flags[["term_id", "is_generic", "has_color", "has_gender", "has_age", "has_number"]],
    on="term_id", how="left",
  )

  def query_bucket(row):
    if row.get("is_generic"):
      return "generic"
    if row.get("has_gender") or row.get("has_age"):
      return "attribute_heavy"
    if row.get("has_number"):
      return "has_number"
    if row.get("has_color"):
      return "has_color"
    return "specific_other"

  neg["query_bucket"] = neg.apply(query_bucket, axis=1)

  safety = (
    neg.groupby(["query_bucket", "negative_type"])["strong_match"]
    .agg(suspicious_rate="mean", count="count")
    .reset_index()
    .sort_values(["query_bucket", "suspicious_rate"])
  )

  print("=== Sorgu türü × strateji (düşük suspicious_rate = daha güvenli) ===")
  display(safety)

  if audit is not None:
    print("\n=== negative_audit_sample.csv ===")
    display(audit.groupby(["negative_type", "negative_confidence"]).size().reset_index(name="count"))